In [3]:
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")  # Or paste your string key directly here if not using env vars
)

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [ ]:
# Q1: 
print(f"Total parsed documents: {len(documents)}")

import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)
print("Search index successfully built with raw documents!")

Total parsed documents: 72
Search index successfully built with raw documents!


In [ ]:
# Q2:
query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query=query,
    filter_dict={}, # No filters needed
    boost_dict={'content': 1.0}, # Standard weight on content
    num_results=1
)

if search_results:
    print("First result filename:", search_results[0]['filename'])
else:
    print("No results found. Make sure your index was fitted correctly!")

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


In [ ]:
# Q3:
def build_context(search_results):
    """
    Constructs the context block using our document schema (content).
    """
    context = ""
    for doc in search_results:
        context += f"Source: {doc['filename']}\nContent:\n{doc['content']}\n\n"
    return context.strip()

def build_prompt(query, context):
    """
    Fuses the context and question into a final prompt template.
    """
    prompt_template = """
You are a helpful assistant. Answer the user QUESTION using the provided CONTEXT from the course lessons.

CONTEXT:
{context}

QUESTION:
{query}
""".strip()
    return prompt_template.format(query=query, context=context)

def custom_rag(query, index, model="gpt-5.4-mini"):
    
    search_results = index.search(query=query, num_results=1)
    
    context = build_context(search_results)
    prompt = build_prompt(query, context)
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    answer = response.choices[0].message.content
    usage = response.usage
    
    return answer, usage

In [ ]:
query = "How does the agentic loop keep calling the model until it stops?"

answer, usage = custom_rag(query=query, index=index)

print(f"Prompt (Input) Tokens: {usage.prompt_tokens}")

Prompt (Input) Tokens: 2314


In [11]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-21 19:22:10--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-21 19:22:10 (24.6 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [ ]:
from rag_helper import RAGBase

class HomeworkRagAssistant(RAGBase):
    def __init__(self, index, llm_client, model='gpt-5.4-mini'):
        # Pass parameters up to the parent constructor
        super().__init__(index, llm_client, model=model)

    def search(self, query, num_results=1):
        # OVERRIDE: Search our new document index without old FAQ filters or boosts
        return self.index.search(
            query=query,
            num_results=num_results
        )

    def build_context(self, search_results):
        # OVERRIDE: Use our new schema ('content') instead of section/question/answer
        lines = []
        for doc in search_results:
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
      
        response = self.llm(prompt)
        
        answer = response.output_text
        
        usage = response.usage 
        
        return answer, usage

In [ ]:
rag_assistant = HomeworkRagAssistant(index=index, llm_client=client, model='gpt-5.4-mini')

query = "How does the agentic loop keep calling the model until it stops?"
answer, usage = rag_assistant.rag(query)

print("--- Usage Statistics ---")
print(usage)

--- Usage Statistics ---
ResponseUsage(input_tokens=2326, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=154, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2480)


In [14]:
#### Q4

In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


In [ ]:
##Q5
import minsearch

chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)
print("Successfully built search index for text chunks!")

Successfully built search index for text chunks!


In [ ]:
chunked_rag_assistant = HomeworkRagAssistant(index=chunk_index, llm_client=client, model='gpt-5.4-mini')

query = "How does the agentic loop keep calling the model until it stops?"
answer, chunked_usage = chunked_rag_assistant.rag(query)

print(f"Chunked Version Input Tokens: {chunked_usage.input_tokens}")

Chunked Version Input Tokens: 531


In [ ]:
###### Q6
def search_course_lessons(query: str) -> str:
    """
    Searches the LLM Zoomcamp course lesson text chunks for relevant context.
    Use this tool to find information about RAG, chunking, or agentic loops.
    """

    results = chunk_index.search(query=query, num_results=3)
    

    context_blocks = []
    for doc in results:
        context_blocks.append(f"Filename: {doc['filename']}\nContent: {doc['content']}")
        
    return "\n\n---\n\n".join(context_blocks)

In [ ]:
from openai import OpenAI
from toyaikit.tools import Tools
# FIX: Import from toyaikit.llm instead of toyaikit.clients
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat import IPythonChatInterface

tools = Tools()

def search_course_lessons(query: str) -> str:
    """
    Searches the LLM Zoomcamp course lesson text chunks for relevant context.
    Use this tool to find information about RAG, chunking, or agentic loops.
    """
    results = chunk_index.search(query=query, num_results=3)
    context_blocks = []
    for doc in results:
        context_blocks.append(f"Filename: {doc['filename']}\nContent: {doc['content']}")
    return "\n\n---\n\n".join(context_blocks)


tools.add_tool(search_course_lessons)

agent_instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
"""

llm_client = OpenAIClient(
    model="gpt-5.4-mini",
    client=OpenAI()
)


runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=agent_instructions,
    chat_interface=IPythonChatInterface(),
    llm_client=llm_client
)


question = "How does the agentic loop work, and how is it different from plain RAG?"
runner.run()


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


KeyboardInterrupt: Interrupted by user